# SSME-Lite on PneumoniaMNIST

Seven released PneumoniaMNIST classifiers are scored on the official test split. AUC is then estimated from 20 visible labels and 400 unlabeled samples.

The interactive report is [pneumoniamnist_report.html](pneumoniamnist_report.html). The 10-seed comparison is in `results/nl20_nu400_m7/`.


## AUC ranking for this estimate

The table below reads the saved file `results/demo_report/metrics.csv`, from the split with `random_state=0`. Intervals are latent-label uncertainty, not standard errors across seeds.


In [1]:
from pathlib import Path
import pandas as pd

def package_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents, here / 'ssme-lite']:
        if (candidate / 'ssme_lite' / '__init__.py').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook from the ssme-lite repository.')

metrics = pd.read_csv(package_root() / 'experiments/pneumoniamnist/results/demo_report/metrics.csv')
auc = metrics.loc[metrics.metric == 'auc', ['model', 'estimate', 'interval_low', 'interval_high']]
auc = auc.sort_values('estimate', ascending=False).reset_index(drop=True)
auc.index = auc.index + 1
print(auc.to_string(float_format=lambda x: f'{x:.4f}'))


             model  estimate  interval_low  interval_high
1  automl_vision_3    0.9853        0.9749         0.9942
2  automl_vision_1    0.9848        0.9753         0.9945
3  automl_vision_2    0.9841        0.9729         0.9933
4      autokeras_2    0.9826        0.9758         0.9904
5      autokeras_3    0.9740        0.9638         0.9836
6      autokeras_1    0.9735        0.9642         0.9823
7    resnet18_28_1    0.9733        0.9627         0.9825


## From inference to the report

A fresh run needs to download the MedMNIST weights and `ai-edge-litert` to read the three AutoML Vision models. The script and the cell below follow the same path.

```bash
python experiments/pneumoniamnist/pneumonia_ssme.py
```


In [ ]:
from pathlib import Path
import sys
from ssme_lite import PredictionMatrixGenerator, SSMEEstimator, SemiSupervisedSplit

def package_root():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents, here / 'ssme-lite']:
        if (candidate / 'ssme_lite' / '__init__.py').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook from the ssme-lite repository.')

ROOT = package_root() / 'experiments/pneumoniamnist'
sys.path.insert(0, str(ROOT))
from pneumonia_models import (
    MODELS, build_models, check_against_official, download_weights, load_test_images, save_scores,
)

images, y = load_test_images()
models = build_models(download_weights())
generator = PredictionMatrixGenerator({name: models[name] for name in MODELS})
scores = generator.fit_transform(images)
check_against_official(models, y)
save_scores(scores, y, generator.model_names_)

splitter = SemiSupervisedSplit(20, 400, random_state=0)
parts = splitter.indices(len(y))
estimator = SSMEEstimator(max_iter=100, early_stopping=False, random_state=0)
estimator.fit(
    scores[parts['estimation']],
    splitter.partial_labels(y, parts),
    model_names=generator.model_names_,
)
report = estimator.report(
    n_draws=100,
    title='PneumoniaMNIST',
    primary_metric='auc',
    sample_ids=parts['estimation'],
)
report.save(ROOT / 'results' / 'demo_report')
print(report.ranking('auc').to_string(index=False))


The seven models are `resnet18_28_1`, three AutoKeras checkpoints, and three AutoML Vision checkpoints. `SemiSupervisedSplit` fixes the split: 20 labels and 400 unlabeled samples in the estimation pool, with the remaining test samples held out. Transfer learning is a separate chain; see `../transfer/transfer_ssme.py`.
